In [1]:
import pandas as pd
import numpy as np
from datasets import load_dataset
import tiktoken
import json
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import re
from groq import Groq
import os
from dotenv import load_dotenv
import random

load_dotenv()  # Load environment variables from a .env file if present
client = Groq(api_key=os.getenv("groq_api_key"))

d:\Work\Github\google-tunix-kaggle\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
# Initialize tokenizer for token counting
encoding = tiktoken.get_encoding("cl100k_base")  # GPT-4 tokenizer

def count_tokens(text):
    """Count tokens in text using tiktoken"""
    if text is None:
        return 0
    return len(encoding.encode(str(text)))


def convert_to_pandas(dataset, batch_size=1000):
    """
    Convert a Hugging Face dataset (streaming or not) to a pandas DataFrame in batches.
    """
    df_list = []
    
    # Hugging Face streaming datasets use iter() for batching
    for batch in dataset.iter(batch_size=batch_size):
        print(f"Processing batch with {len(batch['name'])} records")
        # batch is a dict, convert directly to DataFrame
        df_batch = pd.DataFrame(batch)
        df_list.append(df_batch)
    
    df = pd.concat(df_list, ignore_index=True)
    return df

def sample_hf_data(data, sample_size=1000):

    # Shuffle the dataset (buffer_size controls memory usage)
    shuffled = data.shuffle(buffer_size=sample_size)
    # Take random samples
    sampled_dataset = shuffled.take(sample_size)
    
    return sampled_dataset

### Load Different Code Dataset sources

    - We will collect difference code questions from different sources and collate them
    - Then we will run the questions through oss120B with low/medum reasoning and get the reasoning and response for them
    - Use it for distillation/GRPO later for code domain
    - We have code dataset collected for Algorithmic Code Reasoning, Debugging & Code Repair,Code Completion



    ## Algorithmic Reasoning
        1. deepmind/code_contests
        2. google-research-datasets/mbpp (or mbpp)
        3. openai/humaneval
        4. THUDM/humaneval-x
        5. Algolingo/AlgoLingo-Code-Question-Solutions
        6. qiaojin/CodeQA-10k
        7. codeparrot/codeparrot-clean-train (optional)

    ## Debugging / Repair
        8. princeton-nlp/SWE-bench_Lite
        9. codenet/CodeFlaws
        10. quixbugs/quixbugs

    ## Completions
        11. bigcode/the-stack-v2
        12. bigcode/the-stack-v2-py
        13. codeparrot/codeparrot-clean-valid



#### 1. deepmind/code_contests (For Algorithmic Code Reasoning)

In [19]:

# Stream without downloading - FAST!
code_contest = load_dataset(
    'deepmind/code_contests',    streaming=True
)['train'].select_columns(['name', 'description', 'source', 'difficulty'])

In [20]:
code_contest_sample=sample_hf_data(code_contest, sample_size=5000)
code_contest_train_df = convert_to_pandas(code_contest_sample, batch_size=1000)

Processing batch with 1000 records
Processing batch with 1000 records
Processing batch with 1000 records
Processing batch with 1000 records
Processing batch with 1000 records


In [21]:
code_contest_train_df

,name,description,source,difficulty
0,771_F. Bear and Isomorphic Points,Bearland is a big square on the plane. It cont...,2,12
1,1336_F. Journey,In the wilds far beyond lies the Land of Sacre...,2,12
2,rbx12r02,In a museum there is an empty wall. We can ima...,1,6
3,p01974 Pigeonhole principle,problem\n\nGiven $ N $ different natural numbe...,6,0
4,794_C. Naming Company,Oleg the client and Igor the analyst are good ...,2,9
...,...,...,...,...
4995,p01767 RUPC,Problem statement\n\nA programming contest wil...,6,0
4996,problem-2-2,Julia has a sack full of red and yellow balls ...,3,0
4997,technicalities,Bimal decided to join a non Hill'ffair club be...,3,0
4998,750_F. New Year and Finding Roots,This is an interactive problem. In the interac...,2,12
